# IHC Cell-Type Validation

Validate LLM/Xenium cell-type annotations against pathologist IHC percentages by collapsing LLM labels into the six IHC marker-defined categories and renormalizing both assays across those six categories.

**Cohort selection.** This notebook is cohort-parameterized: it runs against whichever cohort config `COHORT` points at (`bone`, `lymph_node`, or `soft_tissue`). Set the `IHC_COHORT` environment variable before launching, or edit `COHORT` in the parameters cell below. With [papermill](https://papermill.readthedocs.io/) the parameters cell can be overridden per cohort, e.g. `papermill ihc_cell_type_validation.ipynb out_bone.ipynb -p COHORT bone`.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import h5py

CURRENT_DIRECTORY = Path.cwd().resolve()
PIPELINE_ROOT = (
    CURRENT_DIRECTORY.parent
    if (CURRENT_DIRECTORY.parent / "src").exists()
    else CURRENT_DIRECTORY
)
PROJECT_ROOT = PIPELINE_ROOT.parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from src.config import Configuration

# Cohort to validate; override via the IHC_COHORT environment variable or papermill -p COHORT.
COHORT = os.environ.get("IHC_COHORT", "bone")
CONFIG_PATH = PROJECT_ROOT / "configs" / f"{COHORT}.yaml"
IHC_PATH = PROJECT_ROOT / "metadata" / "cell_type_quantifications.csv"
EXCLUDED_SAMPLE_IDS = {"51646-1", "51646-5", "0051646-1", "0051646-5"}

CELL_TYPES = ["T", "B", "Macrophage", "NK", "Neutrophil", "Tumor"]

configuration = Configuration()
configuration.load_from_yaml(CONFIG_PATH)
configuration.create_directories()

VALIDATION_DIRECTORY = configuration.results_directory / "ihc_validation"
FIGURE_DIRECTORY = configuration.figures_directory / "ihc_validation"
VALIDATION_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

processed_anndata_path = configuration.processed_data_directory / "processed.h5ad"
if not processed_anndata_path.exists():
    raise FileNotFoundError(
        f"Processed AnnData not found at {processed_anndata_path}. "
        "Run the pipeline through the annotate stage before this notebook."
    )

configuration.output_directory, processed_anndata_path

(PosixPath('/vast/projects/nzh/wharton/rohit/output/bone'), PosixPath('/vast/projects/nzh/wharton/rohit/output/bone/processed/processed.h5ad'))

## Helper Functions

In [2]:
def normalize_sample_id(value: object) -> str:
    """Normalize sample IDs across IHC, config, and raw Xenium-style names."""

    text = str(value).strip()
    if " " in text:
        text = text.split()[-1]
    match = re.search(r"0*(\d+)-(\d+)$", text)
    if match is None:
        return text
    return f"{int(match.group(1))}-{int(match.group(2))}"


def matches_any(label: str, patterns: tuple[str, ...]) -> bool:
    """Return whether a normalized label matches any regex pattern."""

    return any(re.search(pattern, label) for pattern in patterns)


def collapse_llm_label_to_ihc_bucket(label: object) -> str:
    """Collapse one LLM label into an IHC-comparable bucket."""

    normalized = str(label).strip().lower()
    normalized = normalized.replace("_", " ")

    if matches_any(
        normalized,
        (
            r"\bt[ -]?cell\b",
            r"\bcd3\b",
            r"\bcd4\b",
            r"\bcd8\b",
            r"\btreg\b",
            r"regulatory t",
            r"cytotoxic t",
            r"exhausted t",
        ),
    ):
        return "T"
    if matches_any(
        normalized,
        (
            r"\bb[ -]?cell\b",
            r"\bpax5\b",
            r"\bms4a1\b",
            r"\bcd20\b",
            r"\bcd79",
        ),
    ):
        return "B"
    if matches_any(
        normalized,
        (
            r"neutrophil",
            r"granulocyte",
            r"\bmye\b",
            r"\bmpo\b",
        ),
    ):
        return "Neutrophil"
    if matches_any(
        normalized,
        (
            r"macrophage",
            r"monocyte",
            r"histiocyte",
            r"\bcd68\b",
            r"\bspp1\b",
            r"\btrem2\b",
            r"\bmarco\b",
            r"myeloid",
            r"dendritic",
        ),
    ):
        return "Macrophage"
    if matches_any(
        normalized,
        (
            r"tumou?r",
            r"malignan",
            r"carcinoma",
            r"adenocarcinoma",
            r"epithelial",
            r"luminal",
            r"basal",
            r"prostate",
            r"\bnkx3",
            r"\bklk3\b",
            r"\bkrt",
            r"androgen",
            r"neuroendocrine",
            r"chromogranin",
            r"\bsyp\b",
        ),
    ):
        return "Tumor"
    if matches_any(
        normalized,
        (
            r"\bnk\b",
            r"natural killer",
            r"\bcd56\b",
            r"\bncam1\b",
        ),
    ):
        return "NK"
    return "other"


def normalize_rows(dataframe: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Normalize selected columns so each row sums to one."""

    values = dataframe[columns].astype(float)
    totals = values.sum(axis=1).replace(0, np.nan)
    return values.div(totals, axis=0)



def decode_h5ad_values(values: np.ndarray) -> np.ndarray:
    """Decode H5AD string-like datasets when needed."""

    if values.dtype.kind in {"S", "O"}:
        return np.array([
            value.decode("utf-8") if isinstance(value, bytes) else str(value)
            for value in values
        ], dtype=object)
    return values


def read_h5ad_obs_columns(h5ad_path: Path, columns: list[str]) -> pd.DataFrame:
    """Read selected obs columns directly from an H5AD file."""

    records = {}
    with h5py.File(h5ad_path, "r") as handle:
        obs_group = handle["obs"]
        for column in columns:
            h5ad_column = obs_group[column]
            if isinstance(h5ad_column, h5py.Group):
                encoding_type = h5ad_column.attrs.get("encoding-type")
                if encoding_type == "categorical":
                    categories = decode_h5ad_values(h5ad_column["categories"][:])
                    codes = h5ad_column["codes"][:].astype(int)
                    records[column] = pd.Categorical.from_codes(codes, categories=categories)
                    continue
                raise ValueError(f"Unsupported H5AD obs encoding for {column}: {encoding_type}")
            records[column] = decode_h5ad_values(h5ad_column[:])
    return pd.DataFrame(records)

def concordance_correlation_coefficient(x: pd.Series, y: pd.Series) -> float:
    """Compute Lin's concordance correlation coefficient."""

    paired = pd.concat([x, y], axis=1).dropna()
    if len(paired) < 2:
        return np.nan
    x_values = paired.iloc[:, 0].to_numpy(dtype=float)
    y_values = paired.iloc[:, 1].to_numpy(dtype=float)
    covariance = np.cov(x_values, y_values, ddof=1)[0, 1]
    x_variance = np.var(x_values, ddof=1)
    y_variance = np.var(y_values, ddof=1)
    denominator = x_variance + y_variance + (x_values.mean() - y_values.mean()) ** 2
    if denominator == 0:
        return np.nan
    return float((2 * covariance) / denominator)


def fit_line(x: pd.Series, y: pd.Series) -> tuple[float, float]:
    """Fit y = slope * x + intercept."""

    paired = pd.concat([x, y], axis=1).dropna()
    if len(paired) < 2:
        return np.nan, np.nan
    slope, intercept = np.polyfit(paired.iloc[:, 0], paired.iloc[:, 1], 1)
    return float(slope), float(intercept)

## Load and Normalize IHC Percentages

In [3]:
percent_columns = {
    "T": "T cells (CD3) %",
    "B": "B cells (PAX5) %",
    "Macrophage": "Macrophages (CD68) %",
    "NK": "NK cells (CD56) %",
    "Neutrophil": "Neutrophils (MYE) %",
    "Tumor": "Tumor cells (NKX3.1-CK) %",
}
configured_sample_ids = {
    normalize_sample_id(sample.id) for sample in configuration.samples
} - {normalize_sample_id(sample_id) for sample_id in EXCLUDED_SAMPLE_IDS}

ihc_raw = pd.read_csv(IHC_PATH)
ihc_raw["sample_id"] = ihc_raw["Xenium Region"].map(normalize_sample_id)
ihc_raw["raw_six_type_sum_percent"] = ihc_raw[list(percent_columns.values())].sum(axis=1)
ihc_raw["raw_sum_above_100_percent"] = ihc_raw["raw_six_type_sum_percent"] > 100

ihc_cohort = ihc_raw[ihc_raw["sample_id"].isin(configured_sample_ids)].copy()
if ihc_cohort.empty:
    raise ValueError("No IHC rows matched configured sample IDs.")

ihc_percent = ihc_cohort.rename(columns={value: key for key, value in percent_columns.items()})
ihc_normalized_values = normalize_rows(ihc_percent, CELL_TYPES)
ihc_normalized = pd.concat(
    [
        ihc_percent[["sample_id", "TMA", "Xenium Region", "raw_six_type_sum_percent", "raw_sum_above_100_percent"]].reset_index(drop=True),
        ihc_normalized_values.reset_index(drop=True),
    ],
    axis=1,
)
ihc_normalized.to_csv(VALIDATION_DIRECTORY / "ihc_normalized_percentages.csv", index=False)
ihc_normalized

,sample_id,TMA,Xenium Region,NOTE* CD56 stains neurons/glia and osteoblasts (overestimation in brain and bone),raw_six_type_sum_percent,raw_sum_above_100_percent,T,B,Macrophage,NK,Neutrophil,Tumor
0,50508-4,MET-TMA #2C,7/31/2025 50508-4,NaN,76.223146,False,0.033032,0.000788,0.083392,0.000977,0.059286,0.822524
1,50508-3,MET-TMA #2C,7/31/2025 50508-3,Hemosiderin present,89.788904,False,0.003630,0.000061,0.033092,0.000351,0.012072,0.950794
2,50508-2,MET-TMA #2C,7/31/2025 50508-2,NaN,74.391762,False,0.019606,0.002992,0.074319,0.006087,0.047476,0.849520
3,50702-5,MET-TMA #2B,6/6/2025 50702-5,NaN,79.216619,False,0.029817,0.000779,0.015604,0.044891,0.010555,0.898354
4,50699-1,MET-TMA #2A,6/6/2025 50699-1,NaN,94.061984,False,0.062023,0.010849,0.094028,0.003828,0.006310,0.822962


## Load LLM/Xenium Labels and Collapse to IHC Buckets

In [4]:
required_obs_columns = {"sample_id", "cell_type"}
with h5py.File(processed_anndata_path, "r") as handle:
    available_obs_columns = set(handle["obs"].keys())
missing_obs_columns = required_obs_columns - available_obs_columns
if missing_obs_columns:
    raise ValueError(
        f"AnnData is missing {sorted(missing_obs_columns)}. "
        "Run the pipeline through the annotate stage before this notebook."
    )

transcript_depth_candidates = [
    "total_counts",
    "n_counts",
    "transcript_counts",
    "total_transcripts",
]
transcript_depth_column = next(
    (column for column in transcript_depth_candidates if column in available_obs_columns),
    None,
)
obs_columns = ["sample_id", "cell_type"]
if transcript_depth_column is not None:
    obs_columns.append(transcript_depth_column)

obs = read_h5ad_obs_columns(processed_anndata_path, obs_columns)
obs["sample_id"] = obs["sample_id"].map(normalize_sample_id)
obs = obs[obs["sample_id"].isin(configured_sample_ids)].copy()
obs["ihc_bucket"] = obs["cell_type"].map(collapse_llm_label_to_ihc_bucket)

label_mapping_summary = (
    obs.groupby(["cell_type", "ihc_bucket"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values(["ihc_bucket", "n_cells"], ascending=[True, False])
)
label_mapping_summary.to_csv(
    VALIDATION_DIRECTORY / "xenium_label_mapping_summary.csv",
    index=False,
)
label_mapping_summary


,cell_type,ihc_bucket,n_cells
6,"Macrophage-like cell, CD68-high inflammatory s...",Macrophage,47075
8,"Neutrophil-like cell, MX1-high antimicrobial r...",Neutrophil,54215
12,"T-cell, CD3E-high immune response state",T,8975
1,"Basal epithelial cell, KRT5-high stress state",Tumor,151491
11,"Squamous cell carcinoma, SOX2-OT-high epitheli...",Tumor,140653
2,"Basal epithelial cell, NKX3-1-high stress state",Tumor,121020
9,"Prostatic adenocarcinoma, AR-high proliferativ...",Tumor,63733
10,"Prostatic adenocarcinoma, ERG-high proliferati...",Tumor,24699
7,"Neuroendocrine tumor, CHGA-high secretory state",Tumor,429
0,"Basal epithelial cell, AR-high stress state",Tumor,245


## Normalize Xenium/LLM Proportions Across the Same Six Buckets

In [5]:
bucket_counts = (
    obs.groupby(["sample_id", "ihc_bucket"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)
bucket_count_table = (
    bucket_counts.pivot(index="sample_id", columns="ihc_bucket", values="n_cells")
    .fillna(0)
    .astype(int)
)
for cell_type in CELL_TYPES + ["other"]:
    if cell_type not in bucket_count_table.columns:
        bucket_count_table[cell_type] = 0
bucket_count_table = bucket_count_table[CELL_TYPES + ["other"]]
bucket_count_table["six_bucket_total"] = bucket_count_table[CELL_TYPES].sum(axis=1)
bucket_count_table["all_cells"] = bucket_count_table[CELL_TYPES + ["other"]].sum(axis=1)
bucket_count_table["other_fraction_of_all_cells"] = (
    bucket_count_table["other"] / bucket_count_table["all_cells"].replace(0, np.nan)
)

xenium_normalized_values = bucket_count_table[CELL_TYPES].div(
    bucket_count_table["six_bucket_total"].replace(0, np.nan),
    axis=0,
)
xenium_normalized = pd.concat(
    [bucket_count_table[["six_bucket_total", "all_cells", "other", "other_fraction_of_all_cells"]], xenium_normalized_values],
    axis=1,
).reset_index()
xenium_normalized.to_csv(
    VALIDATION_DIRECTORY / "xenium_normalized_percentages.csv",
    index=False,
)
xenium_normalized

ihc_bucket,sample_id,six_bucket_total,all_cells,other,other_fraction_of_all_cells,T,B,Macrophage,NK,Neutrophil,Tumor
0,50699-1,161231,191837,30606,0.159542,0.015357,0.0,0.081479,0.0,0.016176,0.886988
1,50702-5,126080,157469,31389,0.199334,0.021137,0.0,0.064015,0.0,0.050420,0.864427
2,50508-2,107555,127125,19570,0.153943,0.009669,0.0,0.122765,0.0,0.008554,0.859012
3,50508-3,153109,186443,33334,0.178789,0.002495,0.0,0.029476,0.0,0.285378,0.682651
4,50508-4,64560,116000,51440,0.443448,0.037361,0.0,0.126239,0.0,0.009851,0.826549


## Join Assays and Compute Validation Metrics

In [6]:
ihc_for_join = ihc_normalized.set_index("sample_id")
xenium_for_join = xenium_normalized.set_index("sample_id")
shared_sample_ids = sorted(set(ihc_for_join.index) & set(xenium_for_join.index))
if not shared_sample_ids:
    raise ValueError("No samples were shared between normalized IHC and Xenium tables.")

records = []
for sample_id in shared_sample_ids:
    for cell_type in CELL_TYPES:
        records.append(
            {
                "sample_id": sample_id,
                "cell_type": cell_type,
                "ihc_proportion": ihc_for_join.loc[sample_id, cell_type],
                "xenium_proportion": xenium_for_join.loc[sample_id, cell_type],
                "difference": xenium_for_join.loc[sample_id, cell_type] - ihc_for_join.loc[sample_id, cell_type],
                "ihc_raw_six_type_sum_percent": ihc_for_join.loc[sample_id, "raw_six_type_sum_percent"],
                "ihc_raw_sum_above_100_percent": ihc_for_join.loc[sample_id, "raw_sum_above_100_percent"],
                "xenium_other_fraction_of_all_cells": xenium_for_join.loc[sample_id, "other_fraction_of_all_cells"],
            }
        )
validation_long = pd.DataFrame.from_records(records)
if transcript_depth_column is not None:
    sample_transcript_depth = (
        obs.groupby("sample_id", observed=True)[transcript_depth_column]
        .median()
        .rename("median_transcript_depth")
        .reset_index()
    )
    validation_long = validation_long.merge(sample_transcript_depth, on="sample_id", how="left")
    number_of_depth_bins = min(3, validation_long["median_transcript_depth"].nunique())
    if number_of_depth_bins > 1:
        depth_bin_codes = pd.qcut(
            validation_long["median_transcript_depth"],
            q=number_of_depth_bins,
            labels=False,
            duplicates="drop",
        )
        validation_long["transcript_depth_bin"] = depth_bin_codes.map(
            {0: "low", 1: "medium", 2: "high"}
        )
    else:
        validation_long["transcript_depth_bin"] = "single_bin"
else:
    validation_long["median_transcript_depth"] = np.nan
    validation_long["transcript_depth_bin"] = "unavailable"
validation_long.to_csv(VALIDATION_DIRECTORY / "cell_type_validation_long.csv", index=False)

metric_records = []
for cell_type in CELL_TYPES:
    subset = validation_long[validation_long["cell_type"] == cell_type]
    slope, intercept = fit_line(subset["ihc_proportion"], subset["xenium_proportion"])
    metric_records.append(
        {
            "cell_type": cell_type,
            "n_regions": len(subset),
            "ccc": concordance_correlation_coefficient(subset["ihc_proportion"], subset["xenium_proportion"]),
            "rmse": float(np.sqrt(np.mean(np.square(subset["difference"])))),
            "bias_mean_xenium_minus_ihc": float(subset["difference"].mean()),
            "regression_slope": slope,
            "regression_intercept": intercept,
        }
    )
metrics = pd.DataFrame.from_records(metric_records)
metrics.to_csv(VALIDATION_DIRECTORY / "cell_type_validation_metrics.csv", index=False)

distance_records = []
for sample_id in shared_sample_ids:
    ihc_vector = ihc_for_join.loc[sample_id, CELL_TYPES].to_numpy(dtype=float)
    xenium_vector = xenium_for_join.loc[sample_id, CELL_TYPES].to_numpy(dtype=float)
    distance_records.append(
        {
            "sample_id": sample_id,
            "total_variation_distance": float(0.5 * np.abs(xenium_vector - ihc_vector).sum()),
            "largest_absolute_difference_cell_type": CELL_TYPES[int(np.argmax(np.abs(xenium_vector - ihc_vector)))],
            "largest_absolute_difference": float(np.max(np.abs(xenium_vector - ihc_vector))),
        }
    )
region_distances = pd.DataFrame.from_records(distance_records)
region_distances.to_csv(VALIDATION_DIRECTORY / "region_compositional_distances.csv", index=False)

stratified_mismatch = (
    validation_long.groupby(
        ["cell_type", "ihc_raw_sum_above_100_percent", "transcript_depth_bin"],
        observed=True,
    )
    .agg(
        n_regions=("sample_id", "nunique"),
        mean_difference=("difference", "mean"),
        mean_absolute_difference=("difference", lambda values: values.abs().mean()),
    )
    .reset_index()
)
stratified_mismatch.to_csv(
    VALIDATION_DIRECTORY / "stratified_mismatch_summary.csv",
    index=False,
)

metrics, region_distances, stratified_mismatch

(    cell_type  n_regions  ...  regression_slope  regression_intercept
0           T          5  ...          0.254400              0.009668
1           B          5  ...          0.000000              0.000000
2  Macrophage          5  ...          0.827645              0.035064
3          NK          5  ...          0.000000              0.000000
4  Neutrophil          5  ...         -2.090968              0.130824
5       Tumor          5  ...         -1.152694              1.825421

[6 rows x 7 columns],   sample_id  ...  largest_absolute_difference
0   50508-2  ...                     0.048446
1   50508-3  ...                     0.273307
2   50508-4  ...                     0.049435
3   50699-1  ...                     0.064027
4   50702-5  ...                     0.048411

[5 rows x 4 columns],      cell_type  ...  mean_absolute_difference
0            B  ...                  0.000061
1            B  ...                  0.006921
2            B  ...                  0.000783
3  

## Plots

In [ ]:
sample_order = sorted(validation_long["sample_id"].unique())
color_map = plt.get_cmap("tab10")
sample_colors = {
    sample_id: color_map(index % color_map.N)
    for index, sample_id in enumerate(sample_order)
}
legend_column_count = min(4, max(1, len(sample_order)))
legend_row_count = int(np.ceil(len(sample_order) / legend_column_count))
scatter_bottom_margin = min(0.34, 0.08 + 0.045 * legend_row_count)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8 + 0.35 * legend_row_count),
    sharex=True,
    sharey=True,
)
for axis, cell_type in zip(axes.ravel(), CELL_TYPES):
    subset = validation_long[validation_long["cell_type"] == cell_type]
    point_colors = subset["sample_id"].map(sample_colors)
    axis.scatter(
        subset["ihc_proportion"],
        subset["xenium_proportion"],
        c=list(point_colors),
        s=55,
        edgecolor="black",
        linewidth=0.35,
    )
    axis.plot([0, 1], [0, 1], color="black", linewidth=1, linestyle="--")
    axis.set_title(cell_type)
    axis.set_xlabel("IHC normalized proportion")
    axis.set_ylabel("Xenium/LLM normalized proportion")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor=sample_colors[sample_id],
        markeredgecolor="black",
        markeredgewidth=0.35,
        markersize=7,
        label=sample_id,
    )
    for sample_id in sample_order
]
fig.legend(
    handles=legend_handles,
    title="Sample ID",
    loc="lower center",
    ncol=legend_column_count,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)
fig.tight_layout(rect=[0, scatter_bottom_margin, 1, 1])
fig.savefig(
    FIGURE_DIRECTORY / "ihc_vs_xenium_scatter_by_cell_type.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

fig, axis = plt.subplots(figsize=(8, 4))
axis.bar(metrics["cell_type"], metrics["ccc"])
axis.axhline(0, color="black", linewidth=1)
axis.set_ylabel("Lin's CCC")
axis.set_title("Per-cell-type concordance")
fig.tight_layout()
fig.savefig(FIGURE_DIRECTORY / "cell_type_ccc.png", dpi=300, bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(max(8, 0.45 * len(region_distances)), 4.8))
region_positions = np.arange(len(region_distances))
axis.bar(region_positions, region_distances["total_variation_distance"])
axis.set_xticks(region_positions)
axis.set_xticklabels(region_distances["sample_id"], rotation=90, ha="center", va="top")
axis.set_ylabel("Total variation distance")
axis.set_title("Per-region IHC vs Xenium composition mismatch")
fig.tight_layout()
fig.savefig(
    FIGURE_DIRECTORY / "region_total_variation_distance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

fig, axis = plt.subplots(figsize=(max(10, 0.55 * len(sample_order)), 5.2))
difference_by_region = validation_long.pivot(
    index="sample_id",
    columns="cell_type",
    values="difference",
).loc[sample_order, CELL_TYPES]
absolute_difference_by_region = difference_by_region.abs()
bar_positions = np.arange(len(sample_order))
bar_width = min(0.12, 0.78 / len(CELL_TYPES))
cell_type_colors = plt.get_cmap("Set2")(np.linspace(0, 1, len(CELL_TYPES)))
for index, cell_type in enumerate(CELL_TYPES):
    offsets = bar_positions + (index - (len(CELL_TYPES) - 1) / 2) * bar_width
    axis.bar(
        offsets,
        absolute_difference_by_region[cell_type],
        width=bar_width,
        color=cell_type_colors[index],
        label=cell_type,
    )
axis.set_xticks(bar_positions)
axis.set_xticklabels(sample_order, rotation=90, ha="center", va="top")
axis.set_ylabel("Absolute proportion difference")
axis.set_title("Per-region IHC vs Xenium mismatch by cell type")
axis.legend(title="Cell type", ncol=3, frameon=False)
fig.tight_layout()
fig.savefig(
    FIGURE_DIRECTORY / "region_cell_type_absolute_differences.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Output Files

In [8]:
sorted(path.relative_to(PROJECT_ROOT) for path in VALIDATION_DIRECTORY.glob("*.csv")), sorted(
    path.relative_to(PROJECT_ROOT) for path in FIGURE_DIRECTORY.glob("*.png")
)

([PosixPath('output/bone/analysis/ihc_validation/cell_type_validation_long.csv'), PosixPath('output/bone/analysis/ihc_validation/cell_type_validation_metrics.csv'), PosixPath('output/bone/analysis/ihc_validation/ihc_normalized_percentages.csv'), PosixPath('output/bone/analysis/ihc_validation/region_compositional_distances.csv'), PosixPath('output/bone/analysis/ihc_validation/stratified_mismatch_summary.csv'), PosixPath('output/bone/analysis/ihc_validation/xenium_label_mapping_summary.csv'), PosixPath('output/bone/analysis/ihc_validation/xenium_normalized_percentages.csv')], [PosixPath('output/bone/figures/ihc_validation/cell_type_ccc.png'), PosixPath('output/bone/figures/ihc_validation/ihc_vs_xenium_scatter_by_cell_type.png'), PosixPath('output/bone/figures/ihc_validation/region_cell_type_absolute_differences.png'), PosixPath('output/bone/figures/ihc_validation/region_total_variation_distance.png')])